# Sentiment Analysis using NLP & Machine Learning

This project builds an end-to-end sentiment analysis system using NLP preprocessing techniques and multiple machine learning models. The goal is to classify text into sentiment categories (Positive/Negative/Neutral) and compare model performance.

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\Kiran
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Kiran
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
# LOAD DATASET

df = pd.read_csv("IMDB Dataset.csv")  # change path

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# DATA UNDERSTANDING

print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

df.sample(5)

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
8159,"Yeah, I remember this one! Many years since I ...",positive
14459,End of Days is one of the worst big-budget act...,negative
7361,"Yes, the plot is predictable; yes, there are a...",positive
36752,"Seriously, where is Al Pacino's Oscar nominati...",positive
37563,Artimisia was on late last night. At first I d...,positive


# NLP PREPROCESSING

In [24]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()  # lowercasing
    
    text = re.sub(r"http\S+|www\S+", "", text)  # remove URLs
    
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove punctuation
    
    tokens = text.split()  # tokenization
    
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # lemmatization
    
    return " ".join(tokens)

In [26]:
df['clean_text'] = df['review'].apply(preprocess_text)
df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


# FEATURE ENGINEERING

In [68]:
# Bag of Words

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])

In [69]:
# TF-IDF

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

In [70]:
y = df['sentiment'].map({'positive':1, 'negative':0})

In [71]:
# TRAIN TEST SPLIT

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# MODEL BUILDING

In [73]:
def train_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1 Score": f1_score(y_test, preds)
    }

In [82]:
# Logistic Regression

lr_results = train_model(LogisticRegression(), X_train_tfidf, X_test_tfidf, y_train, y_test)
print(lr_results)

{'Accuracy': 0.8848, 'Precision': 0.8765743073047859, 'Recall': 0.8977971819805517, 'F1 Score': 0.8870588235294118}


In [75]:
# Naive Bayes
nb_results = train_model(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test)

In [76]:
# Decision Tree
dt_results = train_model(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, y_train, y_test)

# MODEL EVALUATION

In [78]:
results = pd.DataFrame({
    "Logistic Regression": lr_results,
    "Naive Bayes": nb_results,
    "Decision Tree": dt_results
}).T

results

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8848,0.876574,0.897797,0.887059
Naive Bayes,0.8460,0.850110,0.843024,0.846552
Decision Tree,0.7143,0.719429,0.709863,0.714614


# COMPARISON & INSIGHTS

### Observations:

- Logistic Regression performed best due to its ability to handle sparse TF-IDF features.
- Naive Bayes worked well with Bag of Words due to probabilistic assumptions.
- Decision Tree showed lower performance due to overfitting on sparse text data.

### Best Model:
Logistic Regression with TF-IDF

### Trade-offs:
- TF-IDF improves accuracy but increases computation
- Naive Bayes is fast but less accurate
- Decision Trees are not ideal for high-dimensional sparse data